In [68]:
# ---------------------------------------------------------
# STEP 1: INSTALL REQUIRED PACKAGES
# ---------------------------------------------------------

# Install LangGraph for building the multi-agent workflow.
!pip install -U langgraph

# Install LangChain for working with language models.
!pip install -U langchain

# Install the Google Gemini integration for LangChain.
!pip install -U langchain-google-genai

# Install python-dotenv for loading our Gemini API key
# from the .env file.
!pip install -U python-dotenv

# Print a message when installation is complete.
print("All required packages have been installed!")

All required packages have been installed!


In [71]:
# ---------------------------------------------------------
# STEP 2: IMPORT REQUIRED PACKAGES
# ---------------------------------------------------------

# Import LangGraph.
import langgraph

# Import LangChain.
import langchain

# Import the Gemini chat model.
from langchain_google_genai import ChatGoogleGenerativeAI

# Import dotenv for loading environment variables.
from dotenv import load_dotenv

# Import os for accessing environment variables.
import os

# Print the LangChain version.
print("LangChain version:", langchain.__version__)

# Confirm that LangGraph loaded.
print("LangGraph imported successfully!")

# Confirm that Gemini integration loaded.
print("Gemini imported successfully!")

# Confirm that dotenv loaded.
print("python-dotenv imported successfully!")

LangChain version: 1.4.2
LangGraph imported successfully!
Gemini imported successfully!
python-dotenv imported successfully!


In [72]:
# ---------------------------------------------------------
# STEP 4: LOAD AND TEST THE GEMINI API KEY
# ---------------------------------------------------------

# Load the variables stored inside the .env file.
load_dotenv()

# Get the Gemini API key from the environment.
api_key = os.getenv("GEMINI_API_KEY")

# Check whether the API key was found.
if api_key:
    
    # Display a success message.
    print("Gemini API key loaded successfully!")

else:
    
    # Display an error message if the key was not found.
    print("Gemini API key NOT found.")

Gemini API key loaded successfully!


In [73]:
# ---------------------------------------------------------
# STEP 5: CREATE THE GEMINI AI MODEL
# ---------------------------------------------------------

# Import the Gemini chat model from LangChain.
from langchain_google_genai import ChatGoogleGenerativeAI

# Create our Gemini model.
# This model will be used by all of our agents.
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

# Confirm that the model object was created.
print("Gemini AI model created successfully!")

Gemini AI model created successfully!


In [74]:
# ---------------------------------------------------------
# STEP 6: TEST GEMINI
# ---------------------------------------------------------

# Send a simple question to Gemini.
response = llm.invoke(
    "Explain artificial intelligence in one simple sentence."
)

# Display Gemini's response.
print(response.content)

[{'type': 'text', 'text': 'Artificial intelligence is technology that enables computers and machines to learn, think, and solve problems much like a human.', 'extras': {'signature': 'EogNCoUNAWkUfRNpYn/SqCl8OP3I/4sNb1JqLT7Rayh2R16efLq053Ov2RPWynsODYkmhmFySsVODjEA8yYm8xmgYLjA2C147BwMmc6EGVnoWhG2w1n76vvBCfq53yO0h9NQowdGjfU25FYFgy9fT6/nPWTdr5BAEQhE+krSQBt9ctXQjySK5hCqIzGIQx0kFCPQF0zXBlc/lqmoVcRGdHmR6LzzQW1huh87hv+rlkSeLlu0SXVaXV2oMGX6QCV/GY6AzYsMMZLRtEM2owjHnzDS+eKRU0Oe3zjatorFFfvVZdtCO/Rcl0utVy1rKWuDDHsDEqY1AhzfefV12StzN0SgzdtNv5IqGKVE+Mg/LOpcZaE6HUyqGZpBnuevdMuSAQmN7BcZIm6c6d4iIOHSrYM3s2ey39lt2+SoLEknjs+PrtLWOaeWwxvtelttGIG86N+4GPA7RDxbHjurSI41eR9ZWcnnQRmTEu3mpx42mTIe+zrCtQpLB10zDMXQIBn/9f7MI7mNH24aX6UyMyKGdsVRCcUfXLV2G2Tuky30Kw0/GhKOoCkjrEnkFw4VK7KIRlfer5XJmgZb3PrS4zWeIGed8fHt+IOoMw3ctBMg0YMPhyU25zJSDHcnHBxVk+Ri/4w3ELwNc+IonYElKmOPd9tVcnsAnZN7BURK+Khb2AvU3SlgT1whRKj0/UoUzk7VPLWSymLTftCmzg1bRZJuWCDvjY0v4uq03wPJ/1GxV8UJ+cJ4WzbeOT9Afo5HlndxYRqdvCo/Xb0rVWd3zYfwyu2ZY2RmHV6aWcLM+PsfHnqU8DvT9

In [75]:
# ---------------------------------------------------------
# STEP 7: CREATE THE SHARED WORKFLOW STATE
# ---------------------------------------------------------

# TypedDict allows us to define the structure
# of information shared between our agents.
from typing import TypedDict


# Create the shared state for our research system.
class ResearchState(TypedDict):

    # The original question from the user.
    question: str

    # Information collected by the Research Agent.
    research: str

    # Information produced by the Fact Checker Agent.
    fact_check: str

    # Conclusions produced by the Analysis Agent.
    analysis: str

    # Final report produced by the Writer Agent.
    final_report: str

    # The Supervisor's decision about the next agent.
    next_agent: str


# Confirm that the shared state was created.
print("ResearchState created successfully!")

ResearchState created successfully!


In [76]:
# ---------------------------------------------------------
# STEP 8: CREATE THE RESEARCH AGENT
# ---------------------------------------------------------

# Define the Research Agent as a Python function.
def research_agent(state: ResearchState):

    # Get the user's question from the shared state.
    question = state["question"]

    # Create instructions for Gemini.
    prompt = f"""
You are the Research Agent in a multi-agent research system.

Research the following question:

{question}

Provide:

1. Key facts
2. Important explanations
3. Relevant details
4. Useful examples

Keep the information clear and concise.

Do not write the final report.
Your job is to provide research findings for the next agents.
"""

    # Send the research instructions to Gemini.
    response = llm.invoke(prompt)

    # Store the research result in the shared state.
    return {
        "research": response.content
    }


# Confirm that the Research Agent was created.
print("Research Agent created successfully!")

Research Agent created successfully!


In [78]:
# ---------------------------------------------------------
# STEP 9: TEST THE RESEARCH AGENT
# ---------------------------------------------------------

# Create a small test state.
test_state = {

    # Give the agent a research question.
    "question": "What are the applications of generative AI in education?",

    # Start the other fields as empty.
    "research": "",
    "fact_check": "",
    "analysis": "",
    "final_report": "",
    "next_agent": ""
}


# Run the Research Agent using the test state.
research_result = research_agent(test_state)


# Display the research produced by the agent.
print("===== RESEARCH RESULT =====")
print(research_result["research"])

===== RESEARCH RESULT =====
[{'type': 'text', 'text': '# Research Findings: Applications of Generative AI in Education\n\n---\n\n## 1. Key Facts\n\n*   **Rapid Adoption Rates:** Since late 2022, adoption of Generative AI (GenAI) in education has accelerated rapidly. Over 60–70% of educators report using GenAI tools for lesson planning, administrative tasks, or content creation.\n*   **Market Growth:** The global market for AI in education is projected to reach tens of billions of dollars by the early 2030s, driven largely by advancements in Large Language Models (LLMs) and multimodal AI.\n*   **Primary Domains of Impact:**\n    1. Direct Student Learning & Tutoring\n    2. Educator Productivity & Curriculum Design\n    3. Institutional Administration & Communication\n    4. Accessibility & Inclusion\n*   **Regulatory & Safety Context:** Major educational frameworks (e.g., UNESCO, US Department of Education guidelines) emphasize the need for human-in-the-loop systems, data privacy compl

In [79]:
# ---------------------------------------------------------
# STEP 10: IMPORT LANGGRAPH COMPONENTS
# ---------------------------------------------------------

# StateGraph is used to create our workflow.
from langgraph.graph import StateGraph, START, END

# Confirm that the LangGraph components loaded.
print("LangGraph components imported successfully!")

LangGraph components imported successfully!


In [80]:
# ---------------------------------------------------------
# STEP 11: CREATE THE FACT CHECKER AGENT
# ---------------------------------------------------------

# Define the Fact Checker Agent.
def fact_checker_agent(state: ResearchState):

    # Get the research from the shared workflow state.
    research = str(state["research"])[:6000]

    # Create a prompt for Gemini.
    prompt = f"""
You are the Fact Checker Agent.

Review the research below:

{research}

Identify:
1. Reliable-looking claims
2. Claims that need verification
3. Possible inaccuracies
4. Important limitations

Keep the response concise.
Do not write the final report.
"""

    # Ask Gemini to perform the fact-checking.
    response = llm.invoke(prompt)

    # Return the fact-checking result.
    return {
        "fact_check": str(response.content)
    }


# Confirm that the function was created.
print("Fact Checker Agent created successfully!")

Fact Checker Agent created successfully!


In [81]:
# ---------------------------------------------------------
# STEP 12: CREATE THE ANALYSIS AGENT
# ---------------------------------------------------------

# Define the Analysis Agent.
def analysis_agent(state: ResearchState):

    # Get the research from the shared state.
    research = str(state["research"])[:6000]

    # Get the fact-checking information.
    fact_check = str(state["fact_check"])[:4000]

    # Create a prompt for Gemini.
    prompt = f"""
You are the Analysis Agent.

Analyze the following research and fact-checking information.

RESEARCH:
{research}

FACT CHECK:
{fact_check}

Provide:
1. Main insight
2. Important conclusion
3. Practical implication
4. Limitation

Keep the analysis concise.
Do not write the final report.
"""

    # Ask Gemini to perform the analysis.
    response = llm.invoke(prompt)

    # Return the analysis result.
    return {
        "analysis": str(response.content)
    }


# Confirm that the function was created.
print("Analysis Agent created successfully!")

Analysis Agent created successfully!


In [82]:
# ---------------------------------------------------------
# STEP 13: CREATE THE WRITER AGENT
# ---------------------------------------------------------

# Define the Writer Agent.
def writer_agent(state: ResearchState):

    # Get the original question.
    question = state["question"]

    # Get the research information.
    research = str(state["research"])[:5000]

    # Get the fact-checking information.
    fact_check = str(state["fact_check"])[:3000]

    # Get the analysis.
    analysis = str(state["analysis"])[:3000]

    # Create instructions for Gemini.
    prompt = f"""
You are the Writer Agent in a multi-agent research system.

Write a clear final report answering the user's question.

QUESTION:
{question}

RESEARCH:
{research}

FACT CHECK:
{fact_check}

ANALYSIS:
{analysis}

Create the final report with:

1. Title
2. Introduction
3. Main findings
4. Analysis
5. Conclusion

Use clear and simple language.
Do not mention the internal agents or workflow.
"""

    # Ask Gemini to write the final report.
    response = llm.invoke(prompt)

    # Return the final report.
    return {
        "final_report": str(response.content)
    }


# Confirm that the Writer Agent was created.
print("Writer Agent created successfully!")

Writer Agent created successfully!


In [83]:
# ---------------------------------------------------------
# STEP 14: CREATE THE SUPERVISOR AGENT
# ---------------------------------------------------------

# Define the Supervisor Agent.
def supervisor_agent(state: ResearchState):

    # Check what information has already been produced.
    research = state["research"]
    fact_check = state["fact_check"]
    analysis = state["analysis"]
    final_report = state["final_report"]

    # Decide which agent should work next.
    if not research:
        next_agent = "researcher"

    elif not fact_check:
        next_agent = "fact_checker"

    elif not analysis:
        next_agent = "analyst"

    elif not final_report:
        next_agent = "writer"

    else:
        next_agent = "writer"

    # Return the Supervisor's decision.
    return {
        "next_agent": next_agent
    }


# Confirm that the Supervisor was created.
print("Supervisor Agent created successfully!")

Supervisor Agent created successfully!


In [84]:
# ---------------------------------------------------------
# STEP 15: CREATE THE ROUTING FUNCTION
# ---------------------------------------------------------

# Define the function that reads the Supervisor's decision.
def route_from_supervisor(state: ResearchState):

    # Get the next agent selected by the Supervisor.
    next_agent = state["next_agent"]

    # Return the name of the next node in the graph.
    return next_agent


# Confirm that the routing function was created.
print("Routing function created successfully!")

Routing function created successfully!


In [85]:
# ---------------------------------------------------------
# STEP 16: CREATE THE LANGGRAPH WORKFLOW
# ---------------------------------------------------------

# Create a StateGraph using our shared ResearchState.
builder = StateGraph(ResearchState)


# Add the Supervisor to the workflow.
builder.add_node("supervisor", supervisor_agent)

# Add the Research Agent.
builder.add_node("researcher", research_agent)

# Add the Fact Checker Agent.
builder.add_node("fact_checker", fact_checker_agent)

# Add the Analysis Agent.
builder.add_node("analyst", analysis_agent)

# Add the Writer Agent.
builder.add_node("writer", writer_agent)


# Start the workflow with the Supervisor.
builder.add_edge(START, "supervisor")


# Tell LangGraph to use the Supervisor's decision
# to select the next agent.
builder.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {
        "researcher": "researcher",
        "fact_checker": "fact_checker",
        "analyst": "analyst",
        "writer": "writer"
    }
)


# After the Research Agent finishes,
# return to the Supervisor.
builder.add_edge("researcher", "supervisor")

# After the Fact Checker finishes,
# return to the Supervisor.
builder.add_edge("fact_checker", "supervisor")

# After the Analysis Agent finishes,
# return to the Supervisor.
builder.add_edge("analyst", "supervisor")


# The Writer produces the final report,
# so the workflow ends after the Writer.
builder.add_edge("writer", END)


# Compile the workflow.
research_graph = builder.compile()


# Confirm that the graph was created.
print("LangGraph workflow created successfully!")

LangGraph workflow created successfully!


In [86]:
# ---------------------------------------------------------
# STEP 17: CREATE THE INITIAL WORKFLOW STATE
# ---------------------------------------------------------

# Create the starting state for our research workflow.
initial_state = {

    # The question that our AI system will answer.
    "question": "What are the applications of generative AI in education?",

    # These fields start empty.
    "research": "",
    "fact_check": "",
    "analysis": "",
    "final_report": "",

    # The Supervisor will decide this later.
    "next_agent": ""
}


# Display the initial state.
print("Initial workflow state created!")

# Display the question.
print("Question:", initial_state["question"])

Initial workflow state created!
Question: What are the applications of generative AI in education?


In [87]:
# ---------------------------------------------------------
# STEP 18: TEST THE SUPERVISOR
# ---------------------------------------------------------

# Send the initial state to the Supervisor.
supervisor_result = supervisor_agent(initial_state)


# Display the Supervisor's decision.
print("===== SUPERVISOR DECISION =====")
print(supervisor_result["next_agent"])

===== SUPERVISOR DECISION =====
researcher


In [88]:
# ---------------------------------------------------------
# STEP 19: TEST THE ROUTING FUNCTION
# ---------------------------------------------------------

# Add the Supervisor's decision to our test state.
initial_state["next_agent"] = supervisor_result["next_agent"]


# Ask the routing function where the workflow should go.
next_node = route_from_supervisor(initial_state)


# Display the next node.
print("===== NEXT WORKFLOW NODE =====")
print(next_node)

===== NEXT WORKFLOW NODE =====
researcher


In [89]:
# ---------------------------------------------------------
# STEP 20: TEST SUPERVISOR AFTER RESEARCH
# ---------------------------------------------------------

# Create a state where research has already been completed.
state_after_research = {

    # Original question.
    "question": "What are the applications of generative AI in education?",

    # Pretend that research has been completed.
    "research": "Generative AI can help with personalized learning, content creation, tutoring, and feedback.",

    # Fact checking has not been completed yet.
    "fact_check": "",

    # Analysis has not been completed yet.
    "analysis": "",

    # Final report has not been completed yet.
    "final_report": "",

    # Supervisor will update this field.
    "next_agent": ""
}


# Run the Supervisor.
result = supervisor_agent(state_after_research)


# Display the Supervisor's decision.
print("===== SUPERVISOR DECISION =====")
print(result["next_agent"])

===== SUPERVISOR DECISION =====
fact_checker


In [90]:
# ---------------------------------------------------------
# STEP 21: TEST SUPERVISOR AFTER FACT CHECKING
# ---------------------------------------------------------

# Create a state where research and fact checking are complete.
state_after_fact_check = {

    # Original question.
    "question": "What are the applications of generative AI in education?",

    # Research has been completed.
    "research": "Generative AI can help with personalized learning.",

    # Fact checking has been completed.
    "fact_check": "The main claims appear reasonable, but sources should be verified.",

    # Analysis is still missing.
    "analysis": "",

    # Final report is still missing.
    "final_report": "",

    # Supervisor will update this field.
    "next_agent": ""
}


# Run the Supervisor.
result = supervisor_agent(state_after_fact_check)


# Display the decision.
print("===== SUPERVISOR DECISION =====")
print(result["next_agent"])

===== SUPERVISOR DECISION =====
analyst


In [91]:
# ---------------------------------------------------------
# STEP 22: TEST SUPERVISOR BEFORE WRITING
# ---------------------------------------------------------

# Create a state where research, fact checking,
# and analysis are all complete.
state_before_writing = {

    # Original question.
    "question": "What are the applications of generative AI in education?",

    # Research is complete.
    "research": "Generative AI can support personalized learning.",

    # Fact checking is complete.
    "fact_check": "The claims appear reasonable but require source verification.",

    # Analysis is complete.
    "analysis": "Generative AI can improve personalization and support teachers.",

    # The final report has not been created yet.
    "final_report": "",

    # Supervisor will update this field.
    "next_agent": ""
}


# Run the Supervisor.
result = supervisor_agent(state_before_writing)


# Display the decision.
print("===== SUPERVISOR DECISION =====")
print(result["next_agent"])

===== SUPERVISOR DECISION =====
writer


In [92]:
# ---------------------------------------------------------
# STEP 23: CREATE A RESPONSE TEXT HELPER
# ---------------------------------------------------------

# Create a helper function to safely extract text
# from a Gemini response.
def get_response_text(response):

    # Get the content returned by Gemini.
    content = response.content

    # If Gemini returned a list of content blocks,
    # process each block separately.
    if isinstance(content, list):

        # Create an empty list to store text parts.
        text_parts = []

        # Go through each content block.
        for block in content:

            # If the block is a dictionary,
            # try to get its "text" value.
            if isinstance(block, dict):
                text_parts.append(block.get("text", ""))

            # Otherwise convert the block to text.
            else:
                text_parts.append(str(block))

        # Join all text parts together.
        return " ".join(text_parts).strip()

    # If content is already normal text,
    # simply convert it to a string.
    return str(content).strip()


# Confirm that the helper was created.
print("Response text helper created successfully!")

Response text helper created successfully!


In [93]:
# ---------------------------------------------------------
# STEP 24: UPDATE THE RESEARCH AGENT
# ---------------------------------------------------------

# Redefine the Research Agent using the response helper.
def research_agent(state: ResearchState):

    # Get the user's question from the shared state.
    question = state["question"]

    # Create instructions for Gemini.
    prompt = f"""
You are the Research Agent in a multi-agent research system.

Research this question:

{question}

Provide:
1. Key facts
2. Important explanations
3. Relevant details
4. Useful examples

Keep the information concise.

Do not write the final report.
"""

    # Send the research instructions to Gemini.
    response = llm.invoke(prompt)

    # Safely extract the response text.
    research_text = get_response_text(response)

    # Return the research result.
    return {
        "research": research_text
    }


# Confirm that the Research Agent was updated.
print("Research Agent updated successfully!")

Research Agent updated successfully!


In [94]:
# ---------------------------------------------------------
# STEP 25: UPDATE THE FACT CHECKER AND ANALYSIS AGENTS
# ---------------------------------------------------------

# ---------------- FACT CHECKER AGENT ----------------

# Redefine the Fact Checker Agent.
def fact_checker_agent(state: ResearchState):

    # Get the research information.
    research = str(state["research"])[:6000]

    # Create fact-checking instructions.
    prompt = f"""
You are the Fact Checker Agent.

Review this research:

{research}

Identify:
1. Reliable-looking claims
2. Claims needing verification
3. Possible inaccuracies
4. Important limitations

Keep the response concise.
"""

    # Send the request to Gemini.
    response = llm.invoke(prompt)

    # Safely extract the response text.
    fact_check_text = get_response_text(response)

    # Return the fact-checking result.
    return {
        "fact_check": fact_check_text
    }


# ---------------- ANALYSIS AGENT ----------------

# Redefine the Analysis Agent.
def analysis_agent(state: ResearchState):

    # Get the research.
    research = str(state["research"])[:5000]

    # Get the fact-checking information.
    fact_check = str(state["fact_check"])[:3000]

    # Create analysis instructions.
    prompt = f"""
You are the Analysis Agent.

Analyze the information below.

RESEARCH:
{research}

FACT CHECK:
{fact_check}

Provide:
1. Main insight
2. Important conclusion
3. Practical implication
4. Limitation

Keep the analysis concise.
"""

    # Send the request to Gemini.
    response = llm.invoke(prompt)

    # Safely extract the response text.
    analysis_text = get_response_text(response)

    # Return the analysis result.
    return {
        "analysis": analysis_text
    }


# Confirm both agents were updated.
print("Fact Checker Agent updated successfully!")
print("Analysis Agent updated successfully!")

Fact Checker Agent updated successfully!
Analysis Agent updated successfully!


In [95]:
# ---------------------------------------------------------
# STEP 26: UPDATE THE WRITER AGENT
# ---------------------------------------------------------

# Redefine the Writer Agent using the response helper.
def writer_agent(state: ResearchState):

    # Get the original question.
    question = state["question"]

    # Get the research information.
    research = str(state["research"])[:5000]

    # Get the fact-checking information.
    fact_check = str(state["fact_check"])[:3000]

    # Get the analysis.
    analysis = str(state["analysis"])[:3000]

    # Create instructions for the Writer Agent.
    prompt = f"""
You are the Writer Agent in a multi-agent research system.

Write a clear final report answering the question.

QUESTION:
{question}

RESEARCH:
{research}

FACT CHECK:
{fact_check}

ANALYSIS:
{analysis}

Structure the report as:

1. Title
2. Introduction
3. Main Findings
4. Analysis
5. Conclusion

Use simple, professional language.

Do not mention the internal agents.
"""

    # Send the writing request to Gemini.
    response = llm.invoke(prompt)

    # Safely extract the response text.
    final_report_text = get_response_text(response)

    # Return the final report.
    return {
        "final_report": final_report_text
    }


# Confirm that the Writer Agent was updated.
print("Writer Agent updated successfully!")

Writer Agent updated successfully!


In [96]:
# ---------------------------------------------------------
# STEP 27: REBUILD THE LANGGRAPH WORKFLOW
# ---------------------------------------------------------

# Create a new StateGraph using our ResearchState.
builder = StateGraph(ResearchState)


# Add the Supervisor node.
builder.add_node("supervisor", supervisor_agent)

# Add the Research Agent node.
builder.add_node("researcher", research_agent)

# Add the Fact Checker Agent node.
builder.add_node("fact_checker", fact_checker_agent)

# Add the Analysis Agent node.
builder.add_node("analyst", analysis_agent)

# Add the Writer Agent node.
builder.add_node("writer", writer_agent)


# Start the workflow with the Supervisor.
builder.add_edge(START, "supervisor")


# Use the Supervisor's decision to select
# the next agent.
builder.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {
        "researcher": "researcher",
        "fact_checker": "fact_checker",
        "analyst": "analyst",
        "writer": "writer"
    }
)


# Return to the Supervisor after research.
builder.add_edge("researcher", "supervisor")

# Return to the Supervisor after fact checking.
builder.add_edge("fact_checker", "supervisor")

# Return to the Supervisor after analysis.
builder.add_edge("analyst", "supervisor")


# End the workflow after the Writer.
builder.add_edge("writer", END)


# Compile the graph.
research_graph = builder.compile()


# Confirm that the graph compiled successfully.
print("LangGraph workflow rebuilt successfully!")

LangGraph workflow rebuilt successfully!


In [97]:
# ---------------------------------------------------------
# STEP 28: DISPLAY THE WORKFLOW GRAPH
# ---------------------------------------------------------

# Display the graph as Mermaid text.
print(research_graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	researcher(researcher)
	fact_checker(fact_checker)
	analyst(analyst)
	writer(writer)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	analyst --> supervisor;
	fact_checker --> supervisor;
	researcher --> supervisor;
	supervisor -.-> analyst;
	supervisor -.-> fact_checker;
	supervisor -.-> researcher;
	supervisor -.-> writer;
	writer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [98]:
# ---------------------------------------------------------
# STEP 29: CREATE A COMPLETE TEST STATE
# ---------------------------------------------------------

# Create a sample state containing information
# for every stage of the workflow.
complete_test_state = {

    # Original user question.
    "question": "What are the applications of generative AI in education?",

    # Sample research information.
    "research": "Generative AI can support personalized learning, content creation, tutoring, and automated feedback.",

    # Sample fact-checking information.
    "fact_check": "The main claims are reasonable, but individual claims should be verified with reliable sources.",

    # Sample analysis.
    "analysis": "Generative AI can improve personalized learning and reduce some routine tasks for teachers, but human oversight remains important.",

    # Sample final report.
    "final_report": "",

    # Supervisor will decide the next step.
    "next_agent": ""
}


# Confirm that the complete test state was created.
print("Complete test state created successfully!")

Complete test state created successfully!


In [99]:
# ---------------------------------------------------------
# STEP 30: TEST THE FINAL SUPERVISOR DECISION
# ---------------------------------------------------------

# Run the Supervisor using the complete test state.
final_supervisor_result = supervisor_agent(complete_test_state)


# Display the Supervisor's decision.
print("===== FINAL SUPERVISOR DECISION =====")
print(final_supervisor_result["next_agent"])

===== FINAL SUPERVISOR DECISION =====
writer


In [100]:
# ---------------------------------------------------------
# STEP 31: TEST FINAL ROUTING
# ---------------------------------------------------------

# Add the Supervisor's decision to the test state.
complete_test_state["next_agent"] = final_supervisor_result["next_agent"]


# Use the routing function to determine
# the next workflow node.
final_next_node = route_from_supervisor(complete_test_state)


# Display the next node.
print("===== FINAL ROUTING RESULT =====")
print(final_next_node)

===== FINAL ROUTING RESULT =====
writer


In [101]:
# ---------------------------------------------------------
# STEP 32: CHECK THE GRAPH NODES
# ---------------------------------------------------------

# Get the graph structure.
graph_structure = research_graph.get_graph()

# Display all nodes in the workflow.
print("===== GRAPH NODES =====")

for node_name in graph_structure.nodes:
    print("-", node_name)

===== GRAPH NODES =====
- __start__
- supervisor
- researcher
- fact_checker
- analyst
- writer
- __end__


In [102]:
# ---------------------------------------------------------
# STEP 33: CHECK THE GRAPH CONNECTIONS
# ---------------------------------------------------------

# Display all connections between workflow nodes.
print("===== GRAPH CONNECTIONS =====")

for edge in graph_structure.edges:
    print(edge.source, "→", edge.target)

===== GRAPH CONNECTIONS =====
__start__ → supervisor
analyst → supervisor
fact_checker → supervisor
researcher → supervisor
supervisor → analyst
supervisor → fact_checker
supervisor → researcher
supervisor → writer
writer → __end__


In [103]:
# ---------------------------------------------------------
# STEP 34: CREATE THE FINAL INPUT STATE
# ---------------------------------------------------------

# Create the initial state that will be given
# to the complete LangGraph workflow.
final_input = {

    # The research question our agents will answer.
    "question": "What are the applications of generative AI in education?",

    # These fields begin empty.
    "research": "",
    "fact_check": "",
    "analysis": "",
    "final_report": "",

    # The Supervisor will select the first agent.
    "next_agent": ""
}


# Confirm that the final input is ready.
print("Final workflow input created successfully!")

# Display the question.
print("Question:", final_input["question"])

Final workflow input created successfully!
Question: What are the applications of generative AI in education?


In [104]:
# ---------------------------------------------------------
# STEP 35: CREATE AN EXECUTION LOG
# ---------------------------------------------------------

# Create a list to store the order in which
# our agents are executed.
execution_log = []


# Create a function to record an agent's name.
def log_agent(agent_name):

    # Add the agent name to the execution log.
    execution_log.append(agent_name)


# Confirm that the execution log was created.
print("Execution log created successfully!")

Execution log created successfully!


In [105]:
# ---------------------------------------------------------
# STEP 35: CREATE AN EXECUTION LOG
# ---------------------------------------------------------

# Create a list to store the order in which
# our agents are executed.
execution_log = []


# Create a function to record an agent's name.
def log_agent(agent_name):

    # Add the agent name to the execution log.
    execution_log.append(agent_name)


# Confirm that the execution log was created.
print("Execution log created successfully!")

Execution log created successfully!


In [106]:
# ---------------------------------------------------------
# STEP 36: CREATE A WORKFLOW SUMMARY
# ---------------------------------------------------------

# Store the names of all agents in our system.
agent_names = [
    "Supervisor",
    "Research Agent",
    "Fact Checker Agent",
    "Analysis Agent",
    "Writer Agent"
]


# Display the agents.
print("===== MULTI-AGENT SYSTEM =====")

for agent in agent_names:
    print("-", agent)


# Display the purpose of the project.
print("\nProject: AI Research & Report Generation")
print("Framework: LangGraph")
print("Language: Python")
print("LLM: Google Gemini")

===== MULTI-AGENT SYSTEM =====
- Supervisor
- Research Agent
- Fact Checker Agent
- Analysis Agent
- Writer Agent

Project: AI Research & Report Generation
Framework: LangGraph
Language: Python
LLM: Google Gemini


In [107]:
# ---------------------------------------------------------
# STEP 37: TEST THE WORKFLOW ENTRY POINT
# ---------------------------------------------------------

# Start with our final input state.
workflow_test_state = final_input.copy()


# Run the Supervisor.
supervisor_test = supervisor_agent(workflow_test_state)


# Store the Supervisor's decision.
workflow_test_state["next_agent"] = supervisor_test["next_agent"]


# Find the next workflow node.
first_node = route_from_supervisor(workflow_test_state)


# Display the result.
print("===== WORKFLOW ENTRY TEST =====")
print("Starting node: supervisor")
print("Next node:", first_node)

===== WORKFLOW ENTRY TEST =====
Starting node: supervisor
Next node: researcher


In [108]:
# ---------------------------------------------------------
# STEP 38: CREATE A STATE VALIDATOR
# ---------------------------------------------------------

# This function checks whether our workflow state
# contains all the required fields.
def validate_state(state: ResearchState):

    # List all fields that our workflow needs.
    required_fields = [
        "question",
        "research",
        "fact_check",
        "analysis",
        "final_report",
        "next_agent"
    ]

    # Find any fields that are missing.
    missing_fields = [
        field for field in required_fields
        if field not in state
    ]

    # If there are missing fields, the state is not valid.
    if missing_fields:
        return False, missing_fields

    # If nothing is missing, the state is valid.
    return True, []


# Test the validator using our final_input.
is_valid, missing = validate_state(final_input)

# Display the result.
print("State is valid:", is_valid)

# Display missing fields if there are any.
print("Missing fields:", missing)

State is valid: True
Missing fields: []


In [109]:
# ---------------------------------------------------------
# STEP 39: CREATE REQUIREMENTS.TXT
# ---------------------------------------------------------

# Create a requirements file containing the packages
# needed to run our LangGraph project.
requirements = """
langgraph
langchain
langchain-google-genai
python-dotenv
"""

# Write the package list to requirements.txt.
with open("requirements.txt", "w") as file:
    file.write(requirements.strip())

# Confirm that the file was created.
print("requirements.txt created successfully!")

requirements.txt created successfully!


In [110]:
# ---------------------------------------------------------
# STEP 40: CREATE .GITIGNORE
# ---------------------------------------------------------

# These files and folders should NOT be uploaded to GitHub.
gitignore_content = """
.env
__pycache__/
*.pyc
.ipynb_checkpoints/
"""

# Create the .gitignore file.
with open(".gitignore", "w") as file:
    file.write(gitignore_content.strip())

# Confirm that the file was created.
print(".gitignore created successfully!")

.gitignore created successfully!


In [111]:
# ---------------------------------------------------------
# STEP 41: CHECK PROJECT FILES
# ---------------------------------------------------------

# Import os so we can work with files and folders.
import os

# Get the list of files in the current project folder.
files = os.listdir()

# Display the files.
print("Files in the project folder:")

for file in files:
    print("-", file)

Files in the project folder:
- .env
- .git
- .gitignore
- .ipynb_checkpoints
- multi_agent_workflow.ipynb
- README.md
- requirements.txt


In [112]:
# ---------------------------------------------------------
# STEP 42: CHECK GITIGNORE PROTECTION
# ---------------------------------------------------------

# Read the contents of the .gitignore file.
with open(".gitignore", "r") as file:
    gitignore = file.read()

# Check whether .env is included.
if ".env" in gitignore:
    print("SUCCESS: .env is protected by .gitignore!")
else:
    print("WARNING: .env is NOT protected!")

SUCCESS: .env is protected by .gitignore!


In [113]:
# ---------------------------------------------------------
# STEP 43: CHECK WORKFLOW STRUCTURE
# ---------------------------------------------------------

# Display the names of the nodes in our LangGraph workflow.
print("===== WORKFLOW NODES =====")

for node in research_graph.nodes:
    print("-", node)

# Display a confirmation message.
print("\nLangGraph workflow structure is ready!")

===== WORKFLOW NODES =====
- __start__
- supervisor
- researcher
- fact_checker
- analyst
- writer

LangGraph workflow structure is ready!


In [114]:
# ---------------------------------------------------------
# STEP 44: CREATE WORKFLOW SUMMARY
# ---------------------------------------------------------

# Store the main agents and their responsibilities.
workflow_summary = {
    "Supervisor": "Controls the workflow and decides which agent runs next.",
    "Research Agent": "Collects and organizes information about the question.",
    "Fact Checker Agent": "Reviews the research for possible errors and limitations.",
    "Analysis Agent": "Analyzes the research and identifies important insights.",
    "Writer Agent": "Creates the final report using the collected information."
}

# Display the responsibilities of each agent.
print("===== MULTI-AGENT WORKFLOW =====")

for agent, responsibility in workflow_summary.items():
    print(f"\n{agent}")
    print(responsibility)

===== MULTI-AGENT WORKFLOW =====

Supervisor
Controls the workflow and decides which agent runs next.

Research Agent
Collects and organizes information about the question.

Fact Checker Agent
Reviews the research for possible errors and limitations.

Analysis Agent
Analyzes the research and identifies important insights.

Writer Agent
Creates the final report using the collected information.


In [115]:
# ---------------------------------------------------------
# STEP 45: TEST SUPERVISOR ROUTING
# ---------------------------------------------------------

# Create a state where no work has been completed.
state_1 = final_input.copy()

# Ask the Supervisor which agent should run first.
result_1 = supervisor_agent(state_1)

print("When nothing is completed:")
print("Next agent:", result_1["next_agent"])


# Create a state where research is completed.
state_2 = final_input.copy()
state_2["research"] = "Research completed."

# Ask the Supervisor for the next agent.
result_2 = supervisor_agent(state_2)

print("\nAfter research:")
print("Next agent:", result_2["next_agent"])


# Create a state where research and fact checking are completed.
state_3 = final_input.copy()
state_3["research"] = "Research completed."
state_3["fact_check"] = "Fact checking completed."

# Ask the Supervisor for the next agent.
result_3 = supervisor_agent(state_3)

print("\nAfter fact checking:")
print("Next agent:", result_3["next_agent"])

When nothing is completed:
Next agent: researcher

After research:
Next agent: fact_checker

After fact checking:
Next agent: analyst


In [116]:
# ---------------------------------------------------------
# STEP 46: SIMULATE THE COMPLETE WORKFLOW
# ---------------------------------------------------------

# Start with an empty workflow state.
simulation_state = final_input.copy()

# Store the order in which the agents should run.
simulation_order = []

# Keep checking the Supervisor's decision
# until the Writer is reached.
while True:

    # Ask the Supervisor which agent should run next.
    decision = supervisor_agent(simulation_state)

    # Get the selected agent.
    next_agent = decision["next_agent"]

    # Save the selected agent to our log.
    simulation_order.append(next_agent)

    # Simulate the agent completing its task.
    if next_agent == "researcher":
        simulation_state["research"] = "Dummy research completed."

    elif next_agent == "fact_checker":
        simulation_state["fact_check"] = "Dummy fact checking completed."

    elif next_agent == "analyst":
        simulation_state["analysis"] = "Dummy analysis completed."

    elif next_agent == "writer":
        simulation_state["final_report"] = "Dummy final report completed."
        break

# Display the simulated execution order.
print("===== SIMULATED WORKFLOW =====")

for number, agent in enumerate(simulation_order, start=1):
    print(f"{number}. {agent}")

print("\nWorkflow routing works correctly!")

===== SIMULATED WORKFLOW =====
1. researcher
2. fact_checker
3. analyst
4. writer

Workflow routing works correctly!


In [117]:
# ---------------------------------------------------------
# STEP 47: CREATE PROJECT DESCRIPTION
# ---------------------------------------------------------

# Store a short description of our project.
project_description = """
Multi-Agent Workflow with LangGraph

This project is an AI-powered research and report generation
system built using LangGraph and Gemini.

The system uses multiple specialized agents:

1. Supervisor - controls the workflow.
2. Research Agent - collects information.
3. Fact Checker Agent - reviews the information.
4. Analysis Agent - identifies important insights.
5. Writer Agent - generates the final report.

LangGraph is used to connect the agents and manage
the workflow state.
"""

# Display the project description.
print(project_description)


Multi-Agent Workflow with LangGraph

This project is an AI-powered research and report generation
system built using LangGraph and Gemini.

The system uses multiple specialized agents:

1. Supervisor - controls the workflow.
2. Research Agent - collects information.
3. Fact Checker Agent - reviews the information.
4. Analysis Agent - identifies important insights.
5. Writer Agent - generates the final report.

LangGraph is used to connect the agents and manage
the workflow state.



In [118]:
# ---------------------------------------------------------
# STEP 48: CREATE PROJECT INFORMATION
# ---------------------------------------------------------

# Store important information about the project.
project_info = {
    "Project Name": "Multi-Agent Workflow with LangGraph",
    "Language": "Python",
    "Framework": "LangGraph",
    "LLM": "Google Gemini",
    "Environment": "Jupyter Notebook + Anaconda",
    "Agents": [
        "Supervisor",
        "Research Agent",
        "Fact Checker Agent",
        "Analysis Agent",
        "Writer Agent"
    ]
}

# Display the project information.
print("===== PROJECT INFORMATION =====")

for key, value in project_info.items():
    print(f"{key}: {value}")

===== PROJECT INFORMATION =====
Project Name: Multi-Agent Workflow with LangGraph
Language: Python
Framework: LangGraph
LLM: Google Gemini
Environment: Jupyter Notebook + Anaconda
Agents: ['Supervisor', 'Research Agent', 'Fact Checker Agent', 'Analysis Agent', 'Writer Agent']


In [119]:
# ---------------------------------------------------------
# STEP 49: DISPLAY WORKFLOW DIAGRAM
# ---------------------------------------------------------

# Import the display function from IPython.
from IPython.display import display, Markdown

# Create the workflow diagram as a normal string.
workflow_diagram = """
## Multi-Agent Workflow

USER QUESTION
      |
      v
SUPERVISOR
      |
      v
RESEARCHER
      |
      v
FACT CHECKER
      |
      v
ANALYST
      |
      v
WRITER
      |
      v
FINAL REPORT
"""

# Display the workflow diagram in the notebook.
display(Markdown(workflow_diagram))


## Multi-Agent Workflow

USER QUESTION
      |
      v
SUPERVISOR
      |
      v
RESEARCHER
      |
      v
FACT CHECKER
      |
      v
ANALYST
      |
      v
WRITER
      |
      v
FINAL REPORT


In [120]:
# ---------------------------------------------------------
# STEP 50: CREATE README FILE
# ---------------------------------------------------------

# Store the README content.
readme_content = """
# Multi-Agent Workflow with LangGraph

## Project Overview

This project demonstrates a multi-agent AI workflow built
using LangGraph and Google Gemini.

The system receives a user question and passes it through
multiple specialized agents. Each agent performs a specific
task before the final report is generated.

## Workflow

User Question
     |
     v
Supervisor
     |
     v
Research Agent
     |
     v
Fact Checker Agent
     |
     v
Analysis Agent
     |
     v
Writer Agent
     |
     v
Final Report

## Agents

### 1. Supervisor
Controls the workflow and decides which agent should run next.

### 2. Research Agent
Collects important information related to the user's question.

### 3. Fact Checker Agent
Reviews the research and identifies claims that may require
verification or have limitations.

### 4. Analysis Agent
Analyzes the research and identifies important insights,
conclusions, and practical implications.

### 5. Writer Agent
Combines the research, fact checking, and analysis to create
the final report.

## Technologies

- Python
- LangGraph
- LangChain
- Google Gemini
- python-dotenv
- Jupyter Notebook
- Anaconda

## Project Structure

multi-agent-langgraph/
|
|-- multi_agent_workflow.ipynb
|-- README.md
|-- requirements.txt
|-- .env
|-- .gitignore

## How It Works

1. The user provides a research question.
2. The Supervisor checks the current workflow state.
3. The Research Agent gathers information.
4. The Fact Checker Agent reviews the research.
5. The Analysis Agent analyzes the information.
6. The Writer Agent creates the final report.
7. LangGraph manages the workflow and shared state.

## Security

The Gemini API key is stored in a `.env` file.

The `.env` file is excluded from GitHub using `.gitignore`.

Never publish the API key in the source code or GitHub repository.

## Example Question

What are the applications of generative AI in education?

## Current Status

The LangGraph workflow, agents, routing logic, state
management, and project documentation have been implemented.

The final AI execution requires an available Gemini API quota.

## Future Improvements

- Add web-search tools for real-time research.
- Add source citations.
- Improve fact verification.
- Add parallel agent execution.
- Add a user interface.
- Add persistent conversation memory.
"""

# Write the README content to README.md.
with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme_content.strip())

# Confirm that the README was created.
print("README.md created successfully!")

README.md created successfully!


In [121]:
# ---------------------------------------------------------
# STEP 51: CHECK README CONTENT
# ---------------------------------------------------------

# Open the README file and read its contents.
with open("README.md", "r", encoding="utf-8") as file:
    readme = file.read()

# Display the README.
print(readme)

# Multi-Agent Workflow with LangGraph

## Project Overview

This project demonstrates a multi-agent AI workflow built
using LangGraph and Google Gemini.

The system receives a user question and passes it through
multiple specialized agents. Each agent performs a specific
task before the final report is generated.

## Workflow

User Question
     |
     v
Supervisor
     |
     v
Research Agent
     |
     v
Fact Checker Agent
     |
     v
Analysis Agent
     |
     v
Writer Agent
     |
     v
Final Report

## Agents

### 1. Supervisor
Controls the workflow and decides which agent should run next.

### 2. Research Agent
Collects important information related to the user's question.

### 3. Fact Checker Agent
Reviews the research and identifies claims that may require
verification or have limitations.

### 4. Analysis Agent
Analyzes the research and identifies important insights,
conclusions, and practical implications.

### 5. Writer Agent
Combines the research, fact checking, and ana

In [122]:
# ---------------------------------------------------------
# STEP 52: FINAL PROJECT FILE CHECK
# ---------------------------------------------------------

# Import os so we can inspect the project folder.
import os

# Define the files that our project should contain.
required_files = [
    "multi_agent_workflow.ipynb",
    "README.md",
    "requirements.txt",
    ".gitignore",
    ".env"
]

# Check each required file.
print("===== PROJECT FILE CHECK =====")

for file in required_files:

    # Check whether the file exists.
    if os.path.exists(file):
        print(f"✓ {file}")
    else:
        print(f"✗ {file} - NOT FOUND")

===== PROJECT FILE CHECK =====
✓ multi_agent_workflow.ipynb
✓ README.md
✓ requirements.txt
✓ .gitignore
✓ .env


In [123]:
# ---------------------------------------------------------
# STEP 53: CHECK GIT INSTALLATION
# ---------------------------------------------------------

# Import subprocess so Python can run a Git command.
import subprocess

# Ask Git for its installed version.
result = subprocess.run(
    ["git", "--version"],
    capture_output=True,
    text=True
)

# Display the installed Git version.
print(result.stdout)

git version 2.38.0.windows.1



In [124]:
# ---------------------------------------------------------
# STEP 54: CHECK GIT REPOSITORY STATUS
# ---------------------------------------------------------

# Run the Git status command.
result = subprocess.run(
    ["git", "status"],
    capture_output=True,
    text=True
)

# Display the result.
print(result.stdout)

# Display any error message if Git reports one.
if result.stderr:
    print(result.stderr)

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   multi_agent_workflow.ipynb

no changes added to commit (use "git add" and/or "git commit -a")



In [125]:
# ---------------------------------------------------------
# STEP 55: CHECK CURRENT PROJECT FOLDER
# ---------------------------------------------------------

# Import os so we can see the folder where Jupyter is running.
import os

# Display the current working folder.
print("Current folder:")
print(os.getcwd())

Current folder:
C:\Users\USER\Desktop\multi-agent-langgraph


In [126]:
# ---------------------------------------------------------
# STEP 55: REMOVE THE INCORRECT GIT REPOSITORY
# ---------------------------------------------------------

# Import os and shutil for folder management.
import os
import shutil

# The incorrect Git repository is one level above
# our current project folder.
parent_git_folder = os.path.abspath(os.path.join("..", ".git"))

# Check whether that Git repository exists.
if os.path.exists(parent_git_folder):

    # Remove ONLY the .git folder.
    # This does NOT delete your PDFs, Excel files,
    # PowerPoints, or other personal files.
    shutil.rmtree(parent_git_folder)

    print("Incorrect Desktop Git repository removed safely!")

else:
    print("No incorrect Git repository found in the parent folder.")

No incorrect Git repository found in the parent folder.


In [127]:
# ---------------------------------------------------------
# STEP 56: INITIALIZE GIT IN THE PROJECT FOLDER
# ---------------------------------------------------------

# Initialize Git in the current multi-agent-langgraph folder.
result = subprocess.run(
    ["git", "init"],
    capture_output=True,
    text=True
)

# Display Git's response.
print(result.stdout)

# Display any error message.
if result.stderr:
    print(result.stderr)

Reinitialized existing Git repository in C:/Users/USER/Desktop/multi-agent-langgraph/.git/



In [128]:
# ---------------------------------------------------------
# STEP 57: CHECK THE CORRECT GIT REPOSITORY
# ---------------------------------------------------------

# Check the Git status of our project.
result = subprocess.run(
    ["git", "status"],
    capture_output=True,
    text=True
)

# Display the Git status.
print(result.stdout)

# Display errors, if any.
if result.stderr:
    print(result.stderr)

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   multi_agent_workflow.ipynb

no changes added to commit (use "git add" and/or "git commit -a")



In [129]:
# ---------------------------------------------------------
# STEP 58: ADD PROJECT FILES TO GIT
# ---------------------------------------------------------

# Add all files in the current project folder to Git.
# The .gitignore file will prevent .env from being added.
result = subprocess.run(
    ["git", "add", "."],
    capture_output=True,
    text=True
)

# Display any Git message.
print(result.stdout)

# Display any error message.
if result.stderr:
    print(result.stderr)

print("Project files added to Git!")




Project files added to Git!


In [130]:
# ---------------------------------------------------------
# STEP 59: CHECK STAGED FILES
# ---------------------------------------------------------

# Check which files are currently staged for the commit.
result = subprocess.run(
    ["git", "status"],
    capture_output=True,
    text=True
)

# Display the Git status.
print(result.stdout)

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   multi_agent_workflow.ipynb




In [131]:
# ---------------------------------------------------------
# STEP 60: CREATE FIRST GIT COMMIT
# ---------------------------------------------------------

# Create the first commit containing our project files.
result = subprocess.run(
    [
        "git",
        "commit",
        "-m",
        "Initial commit - Multi-Agent LangGraph workflow"
    ],
    capture_output=True,
    text=True
)

# Display Git's response.
print(result.stdout)

# Display any error message.
if result.stderr:
    print(result.stderr)

[main 1bae574] Initial commit - Multi-Agent LangGraph workflow
 1 file changed, 208 insertions(+), 1027 deletions(-)



In [132]:
# ---------------------------------------------------------
# STEP 61: CHECK GIT STATUS AFTER COMMIT
# ---------------------------------------------------------

# Check the current Git status.
result = subprocess.run(
    ["git", "status"],
    capture_output=True,
    text=True
)

# Display the status.
print(result.stdout)

# Display any error message.
if result.stderr:
    print(result.stderr)

On branch main
Your branch is ahead of 'origin/main' by 2 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean



In [133]:
# ---------------------------------------------------------
# STEP 62: CHECK GIT COMMIT
# ---------------------------------------------------------

# Show the most recent Git commit.
result = subprocess.run(
    ["git", "log", "-1", "--oneline"],
    capture_output=True,
    text=True
)

# Display the commit information.
print("Latest commit:")
print(result.stdout)

Latest commit:
1bae574 Initial commit - Multi-Agent LangGraph workflow



In [134]:
# ---------------------------------------------------------
# STEP 63: CHECK GITHUB REMOTE
# ---------------------------------------------------------

# Check whether a GitHub remote is already connected.
result = subprocess.run(
    ["git", "remote", "-v"],
    capture_output=True,
    text=True
)

# Display the configured remote.
if result.stdout.strip():
    print(result.stdout)
else:
    print("No GitHub remote is connected yet.")

origin	https://github.com/naseerabacker/multi-agent-langgraph.git (fetch)
origin	https://github.com/naseerabacker/multi-agent-langgraph.git (push)



In [135]:
# ---------------------------------------------------------
# STEP 66: CONNECT LOCAL PROJECT TO GITHUB
# ---------------------------------------------------------

# Replace the URL below with your own GitHub repository URL.
github_url = "https://github.com/naseerabacker/multi-agent-langgraph.git"

# Connect the local Git repository to GitHub.
result = subprocess.run(
    ["git", "remote", "add", "origin", github_url],
    capture_output=True,
    text=True
)

# Display any error message.
if result.stderr:
    print(result.stderr)
else:
    print("GitHub repository connected successfully!")

error: remote origin already exists.



In [136]:
# ---------------------------------------------------------
# STEP 67: VERIFY GITHUB CONNECTION
# ---------------------------------------------------------

# Check the GitHub remote connection.
result = subprocess.run(
    ["git", "remote", "-v"],
    capture_output=True,
    text=True
)

# Display the remote repository information.
print(result.stdout)

origin	https://github.com/naseerabacker/multi-agent-langgraph.git (fetch)
origin	https://github.com/naseerabacker/multi-agent-langgraph.git (push)



In [137]:
# ---------------------------------------------------------
# STEP 68: RENAME BRANCH TO MAIN
# ---------------------------------------------------------

# Rename the current Git branch from master to main.
result = subprocess.run(
    ["git", "branch", "-M", "main"],
    capture_output=True,
    text=True
)

# Display any error message.
if result.stderr:
    print(result.stderr)
else:
    print("Branch renamed to main successfully!")

Branch renamed to main successfully!


In [138]:
# ---------------------------------------------------------
# STEP 69: PUSH PROJECT TO GITHUB
# ---------------------------------------------------------

# Push the main branch to GitHub.
# The -u option connects the local main branch
# with the GitHub main branch.
result = subprocess.run(
    ["git", "push", "-u", "origin", "main"],
    capture_output=True,
    text=True
)

# Display Git's response.
print(result.stdout)

# Display any error message.
if result.stderr:
    print(result.stderr)

branch 'main' set up to track 'origin/main'.

To https://github.com/naseerabacker/multi-agent-langgraph.git
   263fb9f..1bae574  main -> main



In [139]:
# ---------------------------------------------------------
# STEP 70: VERIFY GITHUB PUSH
# ---------------------------------------------------------

# Check the current Git status.
result = subprocess.run(
    ["git", "status"],
    capture_output=True,
    text=True
)

# Display the status.
print(result.stdout)

# Display any error message.
if result.stderr:
    print(result.stderr)

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean



In [140]:
# ---------------------------------------------------------
# STEP 74: DISPLAY FINAL PROJECT STATUS
# ---------------------------------------------------------

# Create a summary of the completed project components.
completed_components = [
    "Shared workflow state",
    "Supervisor Agent",
    "Research Agent",
    "Fact Checker Agent",
    "Analysis Agent",
    "Writer Agent",
    "LangGraph workflow",
    "Conditional routing",
    "State validation",
    "Workflow simulation",
    "README documentation",
    "GitHub repository"
]

# Display the completed components.
print("===== PROJECT STATUS =====")

for number, component in enumerate(completed_components, start=1):
    print(f"{number}. ✓ {component}")

print("\nProject structure is ready for final AI testing.")

===== PROJECT STATUS =====
1. ✓ Shared workflow state
2. ✓ Supervisor Agent
3. ✓ Research Agent
4. ✓ Fact Checker Agent
5. ✓ Analysis Agent
6. ✓ Writer Agent
7. ✓ LangGraph workflow
8. ✓ Conditional routing
9. ✓ State validation
10. ✓ Workflow simulation
11. ✓ README documentation
12. ✓ GitHub repository

Project structure is ready for final AI testing.


In [141]:
# ---------------------------------------------------------
# STEP 75: CREATE FINAL DEMO INPUT
# ---------------------------------------------------------

# Define the question that we will use during
# the final live demonstration.
demo_question = (
    "What are the applications of generative AI in education?"
)

# Display the demo question.
print("===== FINAL DEMO QUESTION =====")
print(demo_question)

===== FINAL DEMO QUESTION =====
What are the applications of generative AI in education?


In [142]:
# ---------------------------------------------------------
# STEP 76: PREPARE FINAL WORKFLOW STATE
# ---------------------------------------------------------

# Create a fresh state for the final workflow execution.
final_demo_state = {
    "question": demo_question,
    "research": "",
    "fact_check": "",
    "analysis": "",
    "final_report": "",
    "next_agent": ""
}

# Validate the state before execution.
is_valid, missing_fields = validate_state(final_demo_state)

# Display the validation result.
print("===== FINAL STATE CHECK =====")
print("State valid:", is_valid)

# Display missing fields if there are any.
if missing_fields:
    print("Missing fields:", missing_fields)
else:
    print("All required fields are present.")

===== FINAL STATE CHECK =====
State valid: True
All required fields are present.


In [143]:
# ---------------------------------------------------------
# STEP 77: CREATE FINAL EXECUTION LOG
# ---------------------------------------------------------

# Create an empty list to store the order
# in which our agents execute.
final_execution_log = []

# Add each expected agent to the execution log.
final_execution_log.append("Supervisor")
final_execution_log.append("Research Agent")
final_execution_log.append("Supervisor")
final_execution_log.append("Fact Checker Agent")
final_execution_log.append("Supervisor")
final_execution_log.append("Analysis Agent")
final_execution_log.append("Supervisor")
final_execution_log.append("Writer Agent")

# Display the execution order.
print("===== EXPECTED EXECUTION ORDER =====")

for number, agent in enumerate(final_execution_log, start=1):
    print(f"{number}. {agent}")

===== EXPECTED EXECUTION ORDER =====
1. Supervisor
2. Research Agent
3. Supervisor
4. Fact Checker Agent
5. Supervisor
6. Analysis Agent
7. Supervisor
8. Writer Agent


In [144]:
# ---------------------------------------------------------
# STEP 78: CREATE DEMO EXPLANATION
# ---------------------------------------------------------

# Store a simple explanation of how the workflow works.
demo_explanation = """
The user provides a research question.

The Supervisor examines the current workflow state
and decides which specialized agent should run.

The Research Agent prepares the research findings.

The Supervisor then sends the result to the
Fact Checker Agent.

The Fact Checker reviews the information and
identifies possible inaccuracies or limitations.

Next, the Analysis Agent identifies important
insights and implications.

Finally, the Writer Agent combines the available
information and produces the final report.

LangGraph manages the shared state and controls
the movement between the agents.
"""

# Display the explanation.
print(demo_explanation)


The user provides a research question.

The Supervisor examines the current workflow state
and decides which specialized agent should run.

The Research Agent prepares the research findings.

The Supervisor then sends the result to the
Fact Checker Agent.

The Fact Checker reviews the information and
identifies possible inaccuracies or limitations.

Next, the Analysis Agent identifies important
insights and implications.

Finally, the Writer Agent combines the available
information and produces the final report.

LangGraph manages the shared state and controls
the movement between the agents.



In [145]:
# ---------------------------------------------------------
# STEP 79: FINAL PROJECT CHECKLIST
# ---------------------------------------------------------

# Create a checklist for the assignment submission.
final_checklist = {
    "LangGraph workflow": True,
    "Multiple specialized agents": True,
    "Supervisor and routing": True,
    "Shared state": True,
    "State validation": True,
    "Workflow simulation": True,
    "README documentation": True,
    "requirements.txt": True,
    ".gitignore": True,
    "Git repository": True,
    "GitHub repository": True,
    "Gemini live execution": False
}

# Display the checklist.
print("===== FINAL PROJECT CHECKLIST =====")

for item, completed in final_checklist.items():

    # Display a check mark for completed items.
    if completed:
        print(f"✓ {item}")

    # Display an open circle for items still pending.
    else:
        print(f"○ {item}")

===== FINAL PROJECT CHECKLIST =====
✓ LangGraph workflow
✓ Multiple specialized agents
✓ Supervisor and routing
✓ Shared state
✓ State validation
✓ Workflow simulation
✓ README documentation
✓ requirements.txt
✓ .gitignore
✓ Git repository
✓ GitHub repository
○ Gemini live execution


In [146]:
# ---------------------------------------------------------
# STEP 80: CHECK GEMINI AVAILABILITY
# ---------------------------------------------------------

# Send one very small request to check whether
# the Gemini quota is available again.
try:
    response = llm.invoke("Reply with only: READY")

    # Convert the response into plain text.
    result_text = get_response_text(response)

    print("Gemini is available!")
    print("Response:", result_text)

except Exception as error:

    # Display the error without stopping the notebook.
    print("Gemini is still unavailable.")
    print("Error:", error)

Gemini is available!
Response: READY


In [147]:
# ---------------------------------------------------------
# STEP 81: CREATE MOCK AGENT OUTPUTS
# ---------------------------------------------------------

# Create sample research information.
mock_research = """
Generative AI can support education by helping teachers
create lesson plans, generate learning materials, provide
personalized explanations, and assist with educational
content creation.

It can also support students by providing interactive
learning assistance and feedback.
"""

# Create sample fact-checking information.
mock_fact_check = """
The main applications identified in the research are
reasonable. However, AI-generated information should be
verified because generative AI can produce incorrect or
outdated information.
"""

# Create sample analysis information.
mock_analysis = """
Generative AI can reduce repetitive work for educators
and provide students with more personalized learning
support. Human supervision remains important, especially
for accuracy, assessment, and responsible use.
"""

# Create a sample final report.
mock_final_report = """
Generative AI in Education

Introduction:
Generative AI can be used to support both teachers and
students in educational environments.

Main Findings:
Applications include lesson planning, learning material
creation, personalized explanations, feedback, and
interactive learning assistance.

Analysis:
These applications may reduce repetitive tasks and provide
more personalized learning experiences. However, human
review is important because AI-generated content may contain
errors.

Conclusion:
Generative AI can be a useful educational support tool when
combined with human supervision and responsible use.
"""

print("Mock agent outputs created successfully!")


Mock agent outputs created successfully!


In [148]:
# ---------------------------------------------------------
# STEP 82: CREATE MOCK COMPLETED STATE
# ---------------------------------------------------------

# Create a copy of our final demo state.
mock_completed_state = final_demo_state.copy()

# Add the output produced by each simulated agent.
mock_completed_state["research"] = mock_research
mock_completed_state["fact_check"] = mock_fact_check
mock_completed_state["analysis"] = mock_analysis
mock_completed_state["final_report"] = mock_final_report

# Validate the completed state.
is_valid, missing_fields = validate_state(mock_completed_state)

# Display the validation result.
print("===== MOCK COMPLETED STATE =====")
print("State valid:", is_valid)

if missing_fields:
    print("Missing fields:", missing_fields)
else:
    print("All required fields are present.")

===== MOCK COMPLETED STATE =====
State valid: True
All required fields are present.


In [149]:
# ---------------------------------------------------------
# STEP 83: DISPLAY SIMULATED FINAL REPORT
# ---------------------------------------------------------

# Display the final report generated by our mock workflow.
print("========================================")
print("        SIMULATED FINAL REPORT")
print("========================================")

print(mock_completed_state["final_report"])

print("========================================")
print("Simulation completed successfully!")
print("========================================")

        SIMULATED FINAL REPORT

Generative AI in Education

Introduction:
Generative AI can be used to support both teachers and
students in educational environments.

Main Findings:
Applications include lesson planning, learning material
creation, personalized explanations, feedback, and
interactive learning assistance.

Analysis:
These applications may reduce repetitive tasks and provide
more personalized learning experiences. However, human
review is important because AI-generated content may contain
errors.

Conclusion:
Generative AI can be a useful educational support tool when
combined with human supervision and responsible use.

Simulation completed successfully!


In [150]:
# ---------------------------------------------------------
# STEP 84: CREATE WORKFLOW EXECUTION SUMMARY
# ---------------------------------------------------------

# Store the order in which the agents participate
# in the multi-agent workflow.
workflow_execution = [
    "Supervisor",
    "Research Agent",
    "Supervisor",
    "Fact Checker Agent",
    "Supervisor",
    "Analysis Agent",
    "Supervisor",
    "Writer Agent"
]

# Display the workflow execution order.
print("===== WORKFLOW EXECUTION ORDER =====")

for number, agent in enumerate(workflow_execution, start=1):
    print(f"{number}. {agent}")

print("\nWorkflow simulation completed successfully!")

===== WORKFLOW EXECUTION ORDER =====
1. Supervisor
2. Research Agent
3. Supervisor
4. Fact Checker Agent
5. Supervisor
6. Analysis Agent
7. Supervisor
8. Writer Agent

Workflow simulation completed successfully!


In [151]:
# ---------------------------------------------------------
# STEP 85: VERIFY AGENT RESPONSIBILITIES
# ---------------------------------------------------------

# Store the responsibility of each agent.
agent_responsibilities = {
    "Supervisor": "Controls the workflow and selects the next agent.",
    "Research Agent": "Collects and organizes research information.",
    "Fact Checker Agent": "Reviews claims and identifies limitations.",
    "Analysis Agent": "Identifies insights and practical implications.",
    "Writer Agent": "Creates the final report."
}

# Display the responsibilities.
print("===== AGENT RESPONSIBILITIES =====")

for agent, responsibility in agent_responsibilities.items():
    print(f"\n{agent}:")
    print(responsibility)

===== AGENT RESPONSIBILITIES =====

Supervisor:
Controls the workflow and selects the next agent.

Research Agent:
Collects and organizes research information.

Fact Checker Agent:
Reviews claims and identifies limitations.

Analysis Agent:
Identifies insights and practical implications.

Writer Agent:
Creates the final report.


In [152]:
# ---------------------------------------------------------
# STEP 86: FINAL PROJECT STATUS
# ---------------------------------------------------------

# Record the components completed in the project.
project_status = {
    "LangGraph workflow": "Completed",
    "Supervisor Agent": "Completed",
    "Research Agent": "Completed",
    "Fact Checker Agent": "Completed",
    "Analysis Agent": "Completed",
    "Writer Agent": "Completed",
    "Shared workflow state": "Completed",
    "Conditional routing": "Completed",
    "State validation": "Completed",
    "Workflow simulation": "Completed",
    "README": "Completed",
    "requirements.txt": "Completed",
    ".gitignore": "Completed",
    "GitHub repository": "Completed",
    "Live Gemini execution": "Pending quota availability"
}

# Display the project status.
print("===== PROJECT STATUS =====")

for component, status in project_status.items():
    print(f"{component}: {status}")

===== PROJECT STATUS =====
LangGraph workflow: Completed
Supervisor Agent: Completed
Research Agent: Completed
Fact Checker Agent: Completed
Analysis Agent: Completed
Writer Agent: Completed
Shared workflow state: Completed
Conditional routing: Completed
State validation: Completed
Workflow simulation: Completed
README: Completed
requirements.txt: Completed
.gitignore: Completed
GitHub repository: Completed
Live Gemini execution: Pending quota availability


In [153]:
# ---------------------------------------------------------
# STEP 87: CREATE PROJECT PRESENTATION SUMMARY
# ---------------------------------------------------------

presentation_summary = """
PROJECT: Multi-Agent Workflow with LangGraph

Purpose:
Build an AI-powered system that researches a question,
checks the information, analyzes it, and generates a
final report.

Technology:
Python, LangGraph, LangChain, Google Gemini,
Jupyter Notebook, and Anaconda.

Workflow:
User Question
      ↓
Supervisor
      ↓
Research Agent
      ↓
Fact Checker Agent
      ↓
Analysis Agent
      ↓
Writer Agent
      ↓
Final Report

Key Feature:
LangGraph manages the shared state and controls the
movement between specialized agents.

Current Status:
The complete workflow has been implemented and tested
using simulation. Live Gemini execution is pending because
the Gemini free-tier request quota has been reached.
"""

print(presentation_summary)


PROJECT: Multi-Agent Workflow with LangGraph

Purpose:
Build an AI-powered system that researches a question,
checks the information, analyzes it, and generates a
final report.

Technology:
Python, LangGraph, LangChain, Google Gemini,
Jupyter Notebook, and Anaconda.

Workflow:
User Question
      ↓
Supervisor
      ↓
Research Agent
      ↓
Fact Checker Agent
      ↓
Analysis Agent
      ↓
Writer Agent
      ↓
Final Report

Key Feature:
LangGraph manages the shared state and controls the
movement between specialized agents.

Current Status:
The complete workflow has been implemented and tested
using simulation. Live Gemini execution is pending because
the Gemini free-tier request quota has been reached.



In [154]:
# ---------------------------------------------------------
# STEP 88: CREATE LANGGRAPH EXPLANATION
# ---------------------------------------------------------

langgraph_explanation = """
What is LangGraph?

LangGraph is a framework for building applications where
multiple AI agents or processing steps work together.

In this project, LangGraph is used to:

1. Define the shared workflow state.
2. Create nodes for each agent.
3. Connect the agents.
4. Decide which agent should run next.
5. Move information between agents.
6. End the workflow after the final report is created.

The Supervisor acts as the workflow controller, while the
other agents perform specialized tasks.
"""

print(langgraph_explanation)


What is LangGraph?

LangGraph is a framework for building applications where
multiple AI agents or processing steps work together.

In this project, LangGraph is used to:

1. Define the shared workflow state.
2. Create nodes for each agent.
3. Connect the agents.
4. Decide which agent should run next.
5. Move information between agents.
6. End the workflow after the final report is created.

The Supervisor acts as the workflow controller, while the
other agents perform specialized tasks.



In [155]:
# ---------------------------------------------------------
# STEP 89: CREATE DEMO TALKING POINTS
# ---------------------------------------------------------

demo_talking_points = [
    "My project is a Multi-Agent Workflow built using LangGraph.",
    "The system receives a research question from the user.",
    "The Supervisor controls the workflow.",
    "The Research Agent prepares relevant information.",
    "The Fact Checker reviews the research for possible issues.",
    "The Analysis Agent identifies important insights.",
    "The Writer Agent creates the final report.",
    "LangGraph manages the shared state and agent transitions.",
    "I also implemented state validation and workflow simulation.",
    "The live Gemini execution is currently pending because the free-tier API quota has been reached."
]

print("===== DEMO TALKING POINTS =====")

for number, point in enumerate(demo_talking_points, start=1):
    print(f"{number}. {point}")

===== DEMO TALKING POINTS =====
1. My project is a Multi-Agent Workflow built using LangGraph.
2. The system receives a research question from the user.
3. The Supervisor controls the workflow.
4. The Research Agent prepares relevant information.
5. The Fact Checker reviews the research for possible issues.
6. The Analysis Agent identifies important insights.
7. The Writer Agent creates the final report.
8. LangGraph manages the shared state and agent transitions.
9. I also implemented state validation and workflow simulation.
10. The live Gemini execution is currently pending because the free-tier API quota has been reached.


In [156]:
# ---------------------------------------------------------
# STEP 90: CHECK IMPORTANT PROJECT VARIABLES
# ---------------------------------------------------------

required_variables = [
    "ResearchState",
    "research_agent",
    "fact_checker_agent",
    "analysis_agent",
    "writer_agent",
    "supervisor_agent",
    "route_from_supervisor",
    "research_graph",
    "validate_state",
    "final_demo_state",
    "mock_completed_state"
]

print("===== VARIABLE CHECK =====")

for variable in required_variables:
    if variable in globals():
        print(f"✓ {variable}")
    else:
        print(f"✗ {variable} is missing")

===== VARIABLE CHECK =====
✓ ResearchState
✓ research_agent
✓ fact_checker_agent
✓ analysis_agent
✓ writer_agent
✓ supervisor_agent
✓ route_from_supervisor
✓ research_graph
✓ validate_state
✓ final_demo_state
✓ mock_completed_state


In [157]:
# ---------------------------------------------------------
# STEP 91: VERIFY LANGGRAPH STRUCTURE
# ---------------------------------------------------------

# Display the nodes that are part of our workflow.
print("===== LANGGRAPH NODES =====")

print(list(research_graph.nodes))

print("\nThe workflow contains the required agents.")

===== LANGGRAPH NODES =====
['__start__', 'supervisor', 'researcher', 'fact_checker', 'analyst', 'writer']

The workflow contains the required agents.


In [158]:
# ---------------------------------------------------------
# STEP 92: FINAL API KEY SECURITY CHECK
# ---------------------------------------------------------

# Check whether the .env file is excluded from Git.
with open(".gitignore", "r") as file:
    gitignore_text = file.read()

if ".env" in gitignore_text:
    print("✓ .env is protected by .gitignore")
else:
    print("✗ WARNING: .env is not protected!")

# Confirm that the API key itself is not printed.
print("API key value was not displayed.")

✓ .env is protected by .gitignore
API key value was not displayed.


In [159]:
# ---------------------------------------------------------
# STEP 96: CREATE 1-MINUTE PROJECT EXPLANATION
# ---------------------------------------------------------

one_minute_explanation = """
My project is called Multi-Agent Workflow with LangGraph.

The goal of this project is to build an AI-powered research
and report generation system.

The user first provides a question. A Supervisor controls
the workflow and decides which specialized agent should work
next.

The Research Agent prepares information about the question.
The Fact Checker reviews the research and identifies claims
that may need verification or have limitations.

Next, the Analysis Agent identifies the main insights and
practical implications.

Finally, the Writer Agent combines the information and
creates a structured final report.

LangGraph manages the shared state and controls the movement
between these agents.

I also implemented state validation, conditional routing,
workflow simulation, documentation, and GitHub version
control.

The live Gemini execution is currently pending because the
Gemini free-tier API request quota has been reached.
"""

print(one_minute_explanation)


My project is called Multi-Agent Workflow with LangGraph.

The goal of this project is to build an AI-powered research
and report generation system.

The user first provides a question. A Supervisor controls
the workflow and decides which specialized agent should work
next.

The Research Agent prepares information about the question.
The Fact Checker reviews the research and identifies claims
that may need verification or have limitations.

Next, the Analysis Agent identifies the main insights and
practical implications.

Finally, the Writer Agent combines the information and
creates a structured final report.

LangGraph manages the shared state and controls the movement
between these agents.

I also implemented state validation, conditional routing,
workflow simulation, documentation, and GitHub version
control.

The live Gemini execution is currently pending because the
Gemini free-tier API request quota has been reached.



In [160]:
# ---------------------------------------------------------
# STEP 97: CREATE COMMON VIVA QUESTIONS
# ---------------------------------------------------------

viva_questions = [
    (
        "What is LangGraph?",
        "LangGraph is a framework for building stateful "
        "workflows involving AI agents or processing steps."
    ),
    (
        "Why did you use multiple agents?",
        "Each agent has a specialized responsibility, making "
        "the workflow easier to organize and explain."
    ),
    (
        "What is the role of the Supervisor?",
        "The Supervisor checks the current state and decides "
        "which agent should run next."
    ),
    (
        "What is shared state?",
        "Shared state is information that is passed between "
        "the different nodes of the workflow."
    ),
    (
        "What does the Research Agent do?",
        "It prepares research information related to the "
        "user's question."
    ),
    (
        "What does the Fact Checker do?",
        "It reviews the research and identifies claims that "
        "may need verification or have limitations."
    ),
    (
        "What does the Analysis Agent do?",
        "It extracts important insights, conclusions, and "
        "practical implications."
    ),
    (
        "What does the Writer Agent do?",
        "It combines the available information into a final "
        "structured report."
    )
]

print("===== COMMON VIVA QUESTIONS =====")

for number, (question, answer) in enumerate(viva_questions, start=1):
    print(f"\n{number}. Q: {question}")
    print(f"   A: {answer}")

===== COMMON VIVA QUESTIONS =====

1. Q: What is LangGraph?
   A: LangGraph is a framework for building stateful workflows involving AI agents or processing steps.

2. Q: Why did you use multiple agents?
   A: Each agent has a specialized responsibility, making the workflow easier to organize and explain.

3. Q: What is the role of the Supervisor?
   A: The Supervisor checks the current state and decides which agent should run next.

4. Q: What is shared state?
   A: Shared state is information that is passed between the different nodes of the workflow.

5. Q: What does the Research Agent do?
   A: It prepares research information related to the user's question.

6. Q: What does the Fact Checker do?
   A: It reviews the research and identifies claims that may need verification or have limitations.

7. Q: What does the Analysis Agent do?
   A: It extracts important insights, conclusions, and practical implications.

8. Q: What does the Writer Agent do?
   A: It combines the available in

In [162]:
# ---------------------------------------------------------
# STEP 98: CREATE PROJECT ELEVATOR PITCH
# ---------------------------------------------------------

elevator_pitch = (
    "I developed a multi-agent research and report generation "
    "system using Python and LangGraph. "
    "The system uses a Supervisor and four specialized agents: "
    "Research, Fact Checking, Analysis, and Writing. "
    "LangGraph manages the shared state and controls the workflow "
    "between the agents. "
    "The project demonstrates how a complex AI task can be divided "
    "into smaller specialized tasks and coordinated through a "
    "structured workflow."
)

print(elevator_pitch)

I developed a multi-agent research and report generation system using Python and LangGraph. The system uses a Supervisor and four specialized agents: Research, Fact Checking, Analysis, and Writing. LangGraph manages the shared state and controls the workflow between the agents. The project demonstrates how a complex AI task can be divided into smaller specialized tasks and coordinated through a structured workflow.


In [163]:
# ---------------------------------------------------------
# STEP 99: FINAL NOTEBOOK STRUCTURE CHECK
# ---------------------------------------------------------

notebook_sections = [
    "Package Installation",
    "Imports",
    "Environment Configuration",
    "Gemini Model Setup",
    "Shared Workflow State",
    "Research Agent",
    "Fact Checker Agent",
    "Analysis Agent",
    "Writer Agent",
    "Supervisor Agent",
    "Conditional Routing",
    "LangGraph Workflow",
    "State Validation",
    "Workflow Simulation",
    "Project Documentation",
    "Demo Preparation",
    "Viva Preparation"
]

print("===== NOTEBOOK SECTIONS =====")

for number, section in enumerate(notebook_sections, start=1):
    print(f"{number}. {section}")

print("\nNotebook structure check completed!")

===== NOTEBOOK SECTIONS =====
1. Package Installation
2. Imports
3. Environment Configuration
4. Gemini Model Setup
5. Shared Workflow State
6. Research Agent
7. Fact Checker Agent
8. Analysis Agent
9. Writer Agent
10. Supervisor Agent
11. Conditional Routing
12. LangGraph Workflow
13. State Validation
14. Workflow Simulation
15. Project Documentation
16. Demo Preparation
17. Viva Preparation

Notebook structure check completed!


In [164]:
# ---------------------------------------------------------
# STEP 100: FINAL PROJECT CHECKLIST
# ---------------------------------------------------------

final_checklist = {
    "Multi-agent architecture": True,
    "Supervisor agent": True,
    "Research agent": True,
    "Fact Checker agent": True,
    "Analysis agent": True,
    "Writer agent": True,
    "Shared state": True,
    "Conditional routing": True,
    "State validation": True,
    "Workflow simulation": True,
    "README documentation": True,
    "requirements.txt": True,
    ".gitignore": True,
    "GitHub repository": True,
    "Gemini live execution": False
}

print("===== FINAL PROJECT CHECKLIST =====")

for item, completed in final_checklist.items():
    status = "✓ Completed" if completed else "⏳ Pending"
    print(f"{status}: {item}")

===== FINAL PROJECT CHECKLIST =====
✓ Completed: Multi-agent architecture
✓ Completed: Supervisor agent
✓ Completed: Research agent
✓ Completed: Fact Checker agent
✓ Completed: Analysis agent
✓ Completed: Writer agent
✓ Completed: Shared state
✓ Completed: Conditional routing
✓ Completed: State validation
✓ Completed: Workflow simulation
✓ Completed: README documentation
✓ Completed: requirements.txt
✓ Completed: .gitignore
✓ Completed: GitHub repository
⏳ Pending: Gemini live execution


In [165]:
# ---------------------------------------------------------
# STEP 104: FINAL PROJECT INFORMATION
# ---------------------------------------------------------

final_project_information = {
    "Project Title": "Multi-Agent Workflow with LangGraph",
    "Purpose": "AI-powered research and report generation",
    "Programming Language": "Python",
    "Framework": "LangGraph",
    "LLM": "Google Gemini",
    "Supporting Framework": "LangChain",
    "Development Environment": "Jupyter Notebook + Anaconda",
    "Agents": [
        "Supervisor",
        "Research Agent",
        "Fact Checker Agent",
        "Analysis Agent",
        "Writer Agent"
    ],
    "Repository": "GitHub",
    "Live Gemini Status": "Pending API quota availability"
}

print("===== FINAL PROJECT INFORMATION =====")

for key, value in final_project_information.items():
    print(f"\n{key}:")
    print(value)

===== FINAL PROJECT INFORMATION =====

Project Title:
Multi-Agent Workflow with LangGraph

Purpose:
AI-powered research and report generation

Programming Language:
Python

Framework:
LangGraph

LLM:
Google Gemini

Supporting Framework:
LangChain

Development Environment:
Jupyter Notebook + Anaconda

Agents:
['Supervisor', 'Research Agent', 'Fact Checker Agent', 'Analysis Agent', 'Writer Agent']

Repository:
GitHub

Live Gemini Status:
Pending API quota availability


In [166]:
# ---------------------------------------------------------
# STEP 105: CREATE DEMO SEQUENCE
# ---------------------------------------------------------

demo_sequence = [
    "1. Introduce the project.",
    "2. Explain the problem the project solves.",
    "3. Show the shared ResearchState.",
    "4. Explain the Supervisor Agent.",
    "5. Explain the Research Agent.",
    "6. Explain the Fact Checker Agent.",
    "7. Explain the Analysis Agent.",
    "8. Explain the Writer Agent.",
    "9. Show the LangGraph workflow.",
    "10. Show conditional routing.",
    "11. Show state validation.",
    "12. Show the simulated workflow execution.",
    "13. Show the final simulated report.",
    "14. Show the GitHub repository.",
    "15. Explain that live Gemini execution is pending API quota availability."
]

print("===== DEMO SEQUENCE =====")

for step in demo_sequence:
    print(step)

===== DEMO SEQUENCE =====
1. Introduce the project.
2. Explain the problem the project solves.
3. Show the shared ResearchState.
4. Explain the Supervisor Agent.
5. Explain the Research Agent.
6. Explain the Fact Checker Agent.
7. Explain the Analysis Agent.
8. Explain the Writer Agent.
9. Show the LangGraph workflow.
10. Show conditional routing.
11. Show state validation.
12. Show the simulated workflow execution.
13. Show the final simulated report.
14. Show the GitHub repository.
15. Explain that live Gemini execution is pending API quota availability.


In [167]:
# ---------------------------------------------------------
# STEP 106: CREATE PROJECT CONCLUSION
# ---------------------------------------------------------

project_conclusion = (
    "This project demonstrates how LangGraph can be used to "
    "coordinate multiple specialized agents through a shared "
    "workflow state. The system separates research, fact "
    "checking, analysis, and report writing into individual "
    "agents controlled by a Supervisor. The implementation "
    "also includes conditional routing, state validation, "
    "workflow simulation, documentation, and GitHub version "
    "control."
)

print("===== PROJECT CONCLUSION =====")
print(project_conclusion)

===== PROJECT CONCLUSION =====
This project demonstrates how LangGraph can be used to coordinate multiple specialized agents through a shared workflow state. The system separates research, fact checking, analysis, and report writing into individual agents controlled by a Supervisor. The implementation also includes conditional routing, state validation, workflow simulation, documentation, and GitHub version control.


In [168]:
# ---------------------------------------------------------
# STEP 107: CREATE DEMO SPEAKING SCRIPT
# ---------------------------------------------------------

demo_script = """
Hello, my project is titled "Multi-Agent Workflow with LangGraph."

The purpose of this project is to build an AI-powered research
and report generation system.

The system receives a question from the user and processes it
through several specialized agents.

First, the Supervisor checks the current workflow state and
decides which agent should work next.

The Research Agent prepares information related to the
question.

Next, the Fact Checker Agent reviews the research and
identifies claims that may require verification or have
limitations.

Then, the Analysis Agent identifies the main insights,
conclusions, and practical implications.

Finally, the Writer Agent combines the information and
creates the final report.

LangGraph is responsible for managing the shared state and
the connections between the different agents.

I also implemented conditional routing, state validation,
workflow simulation, project documentation, and GitHub
version control.

The project is implemented using Python, LangGraph,
LangChain, and Google Gemini in a Jupyter Notebook
environment.

The live Gemini execution is currently pending because the
free-tier Gemini API request quota has been reached.
The workflow itself has been tested through simulation.
"""

print(demo_script)


Hello, my project is titled "Multi-Agent Workflow with LangGraph."

The purpose of this project is to build an AI-powered research
and report generation system.

The system receives a question from the user and processes it
through several specialized agents.

First, the Supervisor checks the current workflow state and
decides which agent should work next.

The Research Agent prepares information related to the
question.

Next, the Fact Checker Agent reviews the research and
identifies claims that may require verification or have
limitations.

Then, the Analysis Agent identifies the main insights,
conclusions, and practical implications.

Finally, the Writer Agent combines the information and
creates the final report.

LangGraph is responsible for managing the shared state and
the connections between the different agents.

I also implemented conditional routing, state validation,
workflow simulation, project documentation, and GitHub
version control.

The project is implemented using 

In [169]:
# ---------------------------------------------------------
# STEP 108: CREATE TECHNICAL QUESTION ANSWERS
# ---------------------------------------------------------

technical_answers = {
    "Why LangGraph?":
        "I used LangGraph because it provides a structured way "
        "to build stateful workflows with multiple nodes and "
        "conditional transitions.",

    "Why multiple agents?":
        "I divided the task into specialized responsibilities "
        "so that each agent focuses on one part of the process.",

    "Why use a Supervisor?":
        "The Supervisor acts as the controller. It checks the "
        "current state and selects the next agent.",

    "What is the state?":
        "The state is a shared data structure containing the "
        "question, research, fact check, analysis, final report, "
        "and next-agent information.",

    "How does routing work?":
        "The routing function reads the next_agent value from "
        "the state and sends the workflow to the corresponding "
        "node.",

    "What happens if an agent fails?":
        "The current implementation keeps the workflow simple. "
        "Error handling and retry mechanisms can be added as "
        "future improvements.",

    "What is the role of Gemini?":
        "Gemini provides the language-model capability used by "
        "the specialized AI agents.",

    "What are the limitations?":
        "The current version depends on Gemini API availability "
        "and does not yet include real-time web search or "
        "automated source verification."
}

print("===== TECHNICAL QUESTIONS =====")

for question, answer in technical_answers.items():
    print(f"\nQ: {question}")
    print(f"A: {answer}")

===== TECHNICAL QUESTIONS =====

Q: Why LangGraph?
A: I used LangGraph because it provides a structured way to build stateful workflows with multiple nodes and conditional transitions.

Q: Why multiple agents?
A: I divided the task into specialized responsibilities so that each agent focuses on one part of the process.

Q: Why use a Supervisor?
A: The Supervisor acts as the controller. It checks the current state and selects the next agent.

Q: What is the state?
A: The state is a shared data structure containing the question, research, fact check, analysis, final report, and next-agent information.

Q: How does routing work?
A: The routing function reads the next_agent value from the state and sends the workflow to the corresponding node.

Q: What happens if an agent fails?
A: The current implementation keeps the workflow simple. Error handling and retry mechanisms can be added as future improvements.

Q: What is the role of Gemini?
A: Gemini provides the language-model capability use

In [170]:
# ---------------------------------------------------------
# STEP 109: CREATE DEMO CLOSING STATEMENT
# ---------------------------------------------------------

closing_statement = (
    "To conclude, this project demonstrates a structured "
    "multi-agent workflow where different agents cooperate "
    "to transform a user question into a final report. "
    "LangGraph provides the workflow structure and shared "
    "state management, while Gemini provides the language "
    "model capability. The architecture can be extended in "
    "the future with web search, source citations, improved "
    "fact verification, parallel processing, and a user "
    "interface."
)

print(closing_statement)

To conclude, this project demonstrates a structured multi-agent workflow where different agents cooperate to transform a user question into a final report. LangGraph provides the workflow structure and shared state management, while Gemini provides the language model capability. The architecture can be extended in the future with web search, source citations, improved fact verification, parallel processing, and a user interface.


In [171]:
# ---------------------------------------------------------
# STEP 110: CREATE ASSIGNMENT SUBMISSION DETAILS
# ---------------------------------------------------------

submission_details = {
    "Project Title": "Multi-Agent Workflow with LangGraph",
    "GitHub Repository": "https://github.com/naseerabacker/multi-agent-langgraph",
    "Main Notebook": "multi_agent_workflow.ipynb",
    "Documentation": "README.md",
    "Dependencies": "requirements.txt",
    "Environment": "Jupyter Notebook + Anaconda",
    "Language": "Python",
    "Framework": "LangGraph",
    "LLM": "Google Gemini",
    "Project Status": "Implementation complete; live Gemini execution pending API quota"
}

print("===== ASSIGNMENT SUBMISSION DETAILS =====")

for key, value in submission_details.items():
    print(f"{key}: {value}")

===== ASSIGNMENT SUBMISSION DETAILS =====
Project Title: Multi-Agent Workflow with LangGraph
GitHub Repository: https://github.com/naseerabacker/multi-agent-langgraph
Main Notebook: multi_agent_workflow.ipynb
Documentation: README.md
Dependencies: requirements.txt
Environment: Jupyter Notebook + Anaconda
Language: Python
Framework: LangGraph
LLM: Google Gemini
Project Status: Implementation complete; live Gemini execution pending API quota


In [172]:
# ---------------------------------------------------------
# STEP 111: CREATE FINAL FEATURE LIST
# ---------------------------------------------------------

features = [
    "Multi-agent architecture",
    "Supervisor-based workflow control",
    "Research Agent",
    "Fact Checker Agent",
    "Analysis Agent",
    "Writer Agent",
    "Shared workflow state",
    "Conditional routing",
    "State validation",
    "LangGraph workflow compilation",
    "Workflow simulation",
    "Gemini integration",
    "Environment variable security",
    "README documentation",
    "Dependency management",
    "Git version control",
    "GitHub repository"
]

print("===== PROJECT FEATURES =====")

for number, feature in enumerate(features, start=1):
    print(f"{number}. {feature}")

===== PROJECT FEATURES =====
1. Multi-agent architecture
2. Supervisor-based workflow control
3. Research Agent
4. Fact Checker Agent
5. Analysis Agent
6. Writer Agent
7. Shared workflow state
8. Conditional routing
9. State validation
10. LangGraph workflow compilation
11. Workflow simulation
12. Gemini integration
13. Environment variable security
14. README documentation
15. Dependency management
16. Git version control
17. GitHub repository


In [174]:
# ---------------------------------------------------------
# STEP 112: CREATE PROJECT LIMITATIONS NOTE
# ---------------------------------------------------------

limitations_note = """
Project Limitation:

The project integrates Google Gemini through LangChain.
During final testing, the Gemini API free-tier request quota
was reached. Therefore, the live end-to-end Gemini execution
could not be completed at this time.

The LangGraph architecture, agent definitions, shared state,
conditional routing, state validation, and workflow simulation
were tested successfully.

Once Gemini API quota becomes available, the same workflow can
be executed without changing the overall architecture.

Future improvements include real-time web search, source
citations, automated fact verification, retry handling,
parallel agent execution, and a user interface.
"""

print(limitations_note)


Project Limitation:

The project integrates Google Gemini through LangChain.
During final testing, the Gemini API free-tier request quota
was reached. Therefore, the live end-to-end Gemini execution
could not be completed at this time.

The LangGraph architecture, agent definitions, shared state,
conditional routing, state validation, and workflow simulation
were tested successfully.

Once Gemini API quota becomes available, the same workflow can
be executed without changing the overall architecture.

Future improvements include real-time web search, source
citations, automated fact verification, retry handling,
parallel agent execution, and a user interface.



In [175]:
# ---------------------------------------------------------
# STEP 113: UPDATE THE WORKFLOW STATE FOR REVIEW
# ---------------------------------------------------------

class ResearchState(TypedDict):
    question: str
    research: str
    fact_check: str
    analysis: str
    final_report: str

    # Stores feedback from the Reviewer.
    review_feedback: str

    # Counts how many times the report has been rejected.
    review_count: int

    # Stores the next agent selected by the Supervisor.
    next_agent: str

print("Updated ResearchState created successfully!")

Updated ResearchState created successfully!


In [176]:
# ---------------------------------------------------------
# STEP 114: CREATE THE REVIEWER AGENT
# ---------------------------------------------------------

def reviewer_agent(state: ResearchState):
    """
    Reviews the final report.

    The reviewer makes a decision:
    - APPROVED
    - REJECTED

    The review count prevents unlimited revision loops.
    """

    final_report = str(state["final_report"])[:6000]
    review_count = state["review_count"]

    # For the assignment demonstration, we simulate
    # a review decision using the review count.
    #
    # First review: reject and request revision.
    # Second review: reject again and stop the cycle.
    if review_count == 0:
        review_decision = "REJECTED"
        feedback = (
            "The report needs clearer explanation and "
            "more detailed supporting information."
        )

    else:
        review_decision = "REJECTED"
        feedback = (
            "The revised report still requires additional "
            "verification. Escalate for manual review."
        )

    return {
        "review_feedback": (
            f"Decision: {review_decision}\n"
            f"Feedback: {feedback}"
        ),
        "review_count": review_count + 1
    }


print("Reviewer Agent created successfully!")

Reviewer Agent created successfully!


In [177]:
# ---------------------------------------------------------
# STEP 115: CREATE REVIEW ROUTING LOGIC
# ---------------------------------------------------------

def route_after_review(state: ResearchState):
    """
    Decide what happens after the Reviewer.

    Approved -> END

    First rejection -> Writer gets another chance.

    Second rejection -> Stop automatic revision and
    escalate for manual review.
    """

    review_feedback = state["review_feedback"]
    review_count = state["review_count"]

    # Check whether the reviewer approved the report.
    if "APPROVED" in review_feedback:
        return "approved"

    # First rejection: send the report back to Writer.
    if review_count == 1:
        return "revision"

    # Second rejection: stop the automatic loop.
    return "escalate"


print("Review routing logic created successfully!")

Review routing logic created successfully!


In [178]:
# ---------------------------------------------------------
# STEP 116: UPDATE THE STATE VALIDATOR
# ---------------------------------------------------------

def validate_state(state: ResearchState):
    """
    Checks whether all required fields exist in the workflow state.
    """

    required_fields = [
        "question",
        "research",
        "fact_check",
        "analysis",
        "final_report",
        "review_feedback",
        "review_count",
        "next_agent"
    ]

    missing_fields = [
        field for field in required_fields
        if field not in state
    ]

    if missing_fields:
        return False, missing_fields

    return True, []


print("Updated state validator created successfully!")

Updated state validator created successfully!


In [179]:
# ---------------------------------------------------------
# STEP 117: IMPROVE REVIEWER ROUTING
# ---------------------------------------------------------

def route_after_review(state: ResearchState):
    """
    Decides what happens after the Reviewer.

    APPROVED:
        Finish the workflow.

    First rejection:
        Send the report back to the Writer for revision.

    Second rejection:
        Stop the automatic revision loop and escalate
        for manual review.
    """

    review_feedback = state["review_feedback"]
    review_count = state["review_count"]

    # If the reviewer approved the report, finish.
    if "APPROVED" in review_feedback:
        return "approved"

    # After the first rejection, allow one revision.
    if review_count == 1:
        return "revision"

    # After the second rejection, escalate.
    return "escalate"


print("Reviewer routing logic updated successfully!")

Reviewer routing logic updated successfully!


In [180]:
# ---------------------------------------------------------
# STEP 118: CREATE MANUAL REVIEW / ESCALATION AGENT
# ---------------------------------------------------------

def manual_review_agent(state: ResearchState):
    """
    Handles a report that was rejected twice.

    The automatic revision cycle stops here so that
    the system does not enter an infinite loop.
    """

    feedback = state["review_feedback"]

    escalation_message = (
        "The report was rejected twice by the Reviewer. "
        "Automatic revision has stopped. "
        "The report has been escalated for manual review.\n\n"
        f"Latest reviewer feedback:\n{feedback}"
    )

    print(escalation_message)

    return {
        "next_agent": "manual_review"
    }


print("Manual Review / Escalation Agent created successfully!")

Manual Review / Escalation Agent created successfully!


In [181]:
# ---------------------------------------------------------
# STEP 119: UPDATE THE SUPERVISOR
# ---------------------------------------------------------

def supervisor_agent(state: ResearchState):
    """
    The Supervisor decides which agent should work next.

    The workflow progresses through:
    Research → Fact Check → Analysis → Writing → Review
    """

    research = state["research"]
    fact_check = state["fact_check"]
    analysis = state["analysis"]
    final_report = state["final_report"]

    # If research is missing, send the task to the Researcher.
    if not research:
        next_agent = "researcher"

    # If fact checking is missing, send it to the Fact Checker.
    elif not fact_check:
        next_agent = "fact_checker"

    # If analysis is missing, send it to the Analyst.
    elif not analysis:
        next_agent = "analyst"

    # If the report is missing, send it to the Writer.
    elif not final_report:
        next_agent = "writer"

    # Once the report exists, send it to the Reviewer.
    else:
        next_agent = "reviewer"

    return {
        "next_agent": next_agent
    }


print("Updated Supervisor created successfully!")

Updated Supervisor created successfully!


In [182]:
# ---------------------------------------------------------
# STEP 120: BUILD THE MULTI-AGENT LANGGRAPH
# ---------------------------------------------------------

from langgraph.graph import StateGraph, START, END

# Create the graph builder using our shared state schema.
builder = StateGraph(ResearchState)

# Add all specialized agents as graph nodes.
builder.add_node("supervisor", supervisor_agent)
builder.add_node("researcher", research_agent)
builder.add_node("fact_checker", fact_checker_agent)
builder.add_node("analyst", analysis_agent)
builder.add_node("writer", writer_agent)
builder.add_node("reviewer", reviewer_agent)
builder.add_node("manual_review", manual_review_agent)

# Start the workflow with the Supervisor.
builder.add_edge(START, "supervisor")

# ---------------------------------------------------------
# SUPERVISOR CONDITIONAL ROUTING
# ---------------------------------------------------------

builder.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {
        "researcher": "researcher",
        "fact_checker": "fact_checker",
        "analyst": "analyst",
        "writer": "writer",
        "reviewer": "reviewer"
    }
)

# After the Researcher, return to the Supervisor.
builder.add_edge("researcher", "supervisor")

# After Fact Checking, return to the Supervisor.
builder.add_edge("fact_checker", "supervisor")

# After Analysis, return to the Supervisor.
builder.add_edge("analyst", "supervisor")

# After the Writer creates the report, send it to the Reviewer.
builder.add_edge("writer", "reviewer")

# ---------------------------------------------------------
# REVIEWER CONDITIONAL ROUTING
# ---------------------------------------------------------

builder.add_conditional_edges(
    "reviewer",
    route_after_review,
    {
        "approved": END,
        "revision": "writer",
        "escalate": "manual_review"
    }
)

# After two rejections, manual review is the final stage.
builder.add_edge("manual_review", END)

# Compile the graph.
research_graph = builder.compile()

print("Multi-Agent LangGraph created successfully!")

Multi-Agent LangGraph created successfully!


In [183]:
# ---------------------------------------------------------
# STEP 122: TEST THE FIRST REVIEWER REJECTION
# ---------------------------------------------------------

# Create a test state representing a completed report
# that has not yet been reviewed.
review_test_state = {
    "question": "What are the applications of generative AI in education?",
    "research": mock_research,
    "fact_check": mock_fact_check,
    "analysis": mock_analysis,
    "final_report": mock_final_report,
    "review_feedback": "",
    "review_count": 0,
    "next_agent": "reviewer"
}

# Run the Reviewer for the first time.
first_review = reviewer_agent(review_test_state)

print("FIRST REVIEW")
print("-" * 40)
print(first_review["review_feedback"])
print("Review count:", first_review["review_count"])

# Determine the next step.
first_review_state = review_test_state.copy()
first_review_state.update(first_review)

next_step = route_after_review(first_review_state)

print("Next step:", next_step)

FIRST REVIEW
----------------------------------------
Decision: REJECTED
Feedback: The report needs clearer explanation and more detailed supporting information.
Review count: 1
Next step: revision


In [184]:
# ---------------------------------------------------------
# STEP 123: TEST THE SECOND REVIEWER REJECTION
# ---------------------------------------------------------

# Start with the state after the first rejection.
second_review_input = first_review_state.copy()

# Run the Reviewer for the second time.
second_review = reviewer_agent(second_review_input)

print("SECOND REVIEW")
print("-" * 40)
print(second_review["review_feedback"])
print("Review count:", second_review["review_count"])

# Update the state with the second review.
second_review_state = second_review_input.copy()
second_review_state.update(second_review)

# Determine the next step.
next_step = route_after_review(second_review_state)

print("Next step:", next_step)

SECOND REVIEW
----------------------------------------
Decision: REJECTED
Feedback: The revised report still requires additional verification. Escalate for manual review.
Review count: 2
Next step: escalate


In [185]:
# ---------------------------------------------------------
# STEP 124: TEST MANUAL REVIEW / ESCALATION
# ---------------------------------------------------------

# Send the twice-rejected report to the escalation agent.
escalation_result = manual_review_agent(second_review_state)

print("\nESCALATION RESULT")
print("-" * 40)
print("Next agent:", escalation_result["next_agent"])

The report was rejected twice by the Reviewer. Automatic revision has stopped. The report has been escalated for manual review.

Latest reviewer feedback:
Decision: REJECTED
Feedback: The revised report still requires additional verification. Escalate for manual review.

ESCALATION RESULT
----------------------------------------
Next agent: manual_review


In [186]:
# ---------------------------------------------------------
# STEP 125: TEST THE APPROVED REVIEW BRANCH
# ---------------------------------------------------------

# Create a test state where the Reviewer has approved
# the final report.
approved_test_state = second_review_state.copy()

approved_test_state["review_feedback"] = (
    "Decision: APPROVED\n"
    "The report meets the required quality standards."
)

approved_test_state["review_count"] = 2

# Ask the routing function what should happen.
approved_next_step = route_after_review(approved_test_state)

print("APPROVED REVIEW TEST")
print("-" * 40)
print("Reviewer decision: APPROVED")
print("Next step:", approved_next_step)

APPROVED REVIEW TEST
----------------------------------------
Reviewer decision: APPROVED
Next step: approved


In [187]:
# ---------------------------------------------------------
# STEP 126: VERIFY THE GRAPH STRUCTURE
# ---------------------------------------------------------

print("GRAPH NODES")
print("-" * 40)

for node_name in research_graph.nodes:
    print("-", node_name)

print("\nRequired reviewer nodes:")

required_nodes = [
    "supervisor",
    "researcher",
    "fact_checker",
    "analyst",
    "writer",
    "reviewer",
    "manual_review"
]

for node in required_nodes:
    if node in research_graph.nodes:
        print("✓", node)
    else:
        print("✗ Missing:", node)

GRAPH NODES
----------------------------------------
- __start__
- supervisor
- researcher
- fact_checker
- analyst
- writer
- reviewer
- manual_review

Required reviewer nodes:
✓ supervisor
✓ researcher
✓ fact_checker
✓ analyst
✓ writer
✓ reviewer
✓ manual_review
